# DAGMA test on PeMS traffic datasets

این notebook همان تست DAGMA است، ولی برای دیتاست‌های ترافیکی مقاله مثل `PEMSD3`, `PEMSD4`, `PEMSD7`, `PEMSD8` آماده شده است.

- اگر `linear.py` را هنوز تغییر نداده باشی، فقط `Standard DAGMA` اجرا می‌شود.
- اگر patch مربوط به `use_shrinkage` را به `DagmaLinear.fit` اضافه کرده باشی، حالت `SCGL-DAGMA` هم اجرا می‌شود.


## 1. Import packages

In [ ]:
import os
import inspect
from timeit import default_timer as timer

import numpy as np

from dagma.linear import DagmaLinear


## 2. Configuration

فقط این بخش را برای مسیر دیتاست خودت عوض کن. برای شروع بهتر است `PEMSD8` را اجرا کنی چون سبک‌تر است.

In [ ]:
DATASET_NAME = "PEMSD8"
FILE_PATH = "./data/PEMSD8.npz"

# If your .npz has a specific key, write it here. Otherwise keep None.
DATA_KEY = None

# PeMS data are usually 5-minute intervals: 288 time steps per day.
TRAIN_DAYS = 14
STEPS_PER_DAY = 288
FEATURE_INDEX = 0

# For a quick debug run, set MAX_NODES to something like 50.
# For the real experiment, keep it None.
MAX_NODES = None

LAMBDA1 = 0.03
W_THRESHOLD = 0.3

# Run shrinkage version only if your patched linear.py supports use_shrinkage.
RUN_SCGL = True


## 3. Load and prepare traffic data

In [ ]:
def load_traffic_array(file_path, data_key=None):
    """Load traffic data from npz, npy, csv, or h5/hdf5."""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".npz":
        obj = np.load(file_path)
        print("Available keys:", obj.files)

        if data_key is not None:
            data = obj[data_key]
        elif "data" in obj.files:
            data = obj["data"]
        else:
            data = obj[obj.files[0]]

    elif ext == ".npy":
        data = np.load(file_path)

    elif ext == ".csv":
        data = np.loadtxt(file_path, delimiter=",")

    elif ext in [".h5", ".hdf5"]:
        import pandas as pd
        data = pd.read_hdf(file_path).values

    else:
        raise ValueError(f"Unsupported file extension: {ext}")

    return np.asarray(data)


def prepare_dagma_input(
    data,
    train_days=14,
    steps_per_day=288,
    feature_index=0,
    max_nodes=None,
):
    """
    Convert traffic time series to DAGMA input.

    DAGMA expects:
        X.shape = (n_samples, n_nodes)
    """
    data = np.asarray(data)

    if data.ndim == 3:
        # common PeMS format: (time, nodes, features)
        X = data[:, :, feature_index]
    elif data.ndim == 2:
        X = data
    else:
        raise ValueError(f"Unexpected data shape: {data.shape}")

    X = X.astype(np.float64)

    n_samples = train_days * steps_per_day
    X = X[:min(n_samples, X.shape[0])]

    if max_nodes is not None:
        X = X[:, :max_nodes]

    # Remove constant sensors if any
    std = X.std(axis=0)
    keep = std > 1e-8
    if keep.sum() < X.shape[1]:
        print(f"Removed {X.shape[1] - keep.sum()} constant sensors.")
        X = X[:, keep]

    # Standardize before DAGMA
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)

    return X


## 4. Run DAGMA initializer

In [ ]:
def fit_dagma(X, use_shrinkage=False):
    """
    Run DagmaLinear on X.

    If the installed/patched DagmaLinear.fit supports use_shrinkage, this function passes it.
    Otherwise, it runs the original DAGMA when use_shrinkage=False and raises a clear error when True.
    """
    model = DagmaLinear(loss_type="l2", verbose=True)
    fit_signature = inspect.signature(model.fit)
    supports_shrinkage = "use_shrinkage" in fit_signature.parameters

    start = timer()

    if supports_shrinkage:
        W_est = model.fit(
            X,
            lambda1=LAMBDA1,
            w_threshold=W_THRESHOLD,
            use_shrinkage=use_shrinkage,
        )
    else:
        if use_shrinkage:
            raise RuntimeError(
                "This DagmaLinear.fit does not support use_shrinkage yet. "
                "Patch linear.py first, then rerun this cell."
            )
        W_est = model.fit(
            X,
            lambda1=LAMBDA1,
            w_threshold=W_THRESHOLD,
        )

    elapsed = timer() - start
    A_init = (np.abs(W_est) > 0).astype(np.float64)

    return model, W_est, A_init, elapsed


def print_result(method_name, model, W_est, A_init, elapsed):
    n_edges = int(A_init.sum())
    p = A_init.shape[0]
    density = n_edges / max(p * (p - 1), 1)

    print("
" + "=" * 70)
    print(method_name)
    print("=" * 70)
    print(f"time: {elapsed:.4f}s")
    print(f"W_est shape: {W_est.shape}")
    print(f"A_init shape: {A_init.shape}")
    print(f"edges: {n_edges}")
    print(f"density: {density:.6f}")
    print(f"h_final: {model.h_final:.4e}")
    print(f"score_final: {model.score_final:.4e}")

    for attr in ["cond_sample", "cond_shrink", "shrinkage_rho"]:
        if hasattr(model, attr):
            value = getattr(model, attr)
            if value is not None:
                print(f"{attr}: {value:.4e}" if isinstance(value, float) else f"{attr}: {value}")


## 5. Load selected dataset

In [ ]:
data = load_traffic_array(FILE_PATH, data_key=DATA_KEY)
print(f"Dataset: {DATASET_NAME}")
print(f"Raw data shape: {data.shape}")

X = prepare_dagma_input(
    data,
    train_days=TRAIN_DAYS,
    steps_per_day=STEPS_PER_DAY,
    feature_index=FEATURE_INDEX,
    max_nodes=MAX_NODES,
)

print(f"DAGMA input X shape: {X.shape}")
print(f"p/n ratio: {X.shape[1] / X.shape[0]:.4f}")


## 6. Standard DAGMA

In [ ]:
model_dagma, W_dagma, A_dagma, time_dagma = fit_dagma(X, use_shrinkage=False)
print_result("Standard DAGMA", model_dagma, W_dagma, A_dagma, time_dagma)

np.savez(
    f"{DATASET_NAME}_dagma_init.npz",
    W_est=W_dagma,
    A_init=A_dagma,
)
print(f"Saved: {DATASET_NAME}_dagma_init.npz")


## 7. SCGL-DAGMA with shrinkage covariance

این cell فقط وقتی اجرا می‌شود که `linear.py` را patch کرده باشی و `DagmaLinear.fit(..., use_shrinkage=True)` وجود داشته باشد.

In [ ]:
if RUN_SCGL:
    model_scgl, W_scgl, A_scgl, time_scgl = fit_dagma(X, use_shrinkage=True)
    print_result("SCGL-DAGMA", model_scgl, W_scgl, A_scgl, time_scgl)

    np.savez(
        f"{DATASET_NAME}_scgl_init.npz",
        W_est=W_scgl,
        A_init=A_scgl,
    )
    print(f"Saved: {DATASET_NAME}_scgl_init.npz")
else:
    print("RUN_SCGL=False, skipped.")


## 8. Quick dataset switch examples

برای دیتاست‌های دیگر فقط `DATASET_NAME` و `FILE_PATH` را در cell دوم عوض کن:

```python
DATASET_NAME = "PEMSD4"
FILE_PATH = "./data/PEMSD4.npz"
```

```python
DATASET_NAME = "PEMSD7"
FILE_PATH = "./data/PEMSD7.npz"
```
